In [0]:
dbutils.widgets.text("p_batch_id", "", "Batch ID")

v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise ValueError("Enter a batch ID")

In [0]:
%run ../00_common/01_environment_config

In [0]:
%run ../00_common/04_gold_helpers

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
constructors_silver_path = "s3://bizf1project123/Silver/Kalashri_increment/constructors"

nationality_region_gold_path = "s3://bizf1project123/Gold/Kalashri_increment/ref_nationality_region"

gold_path = "s3://bizf1project123/Gold/Kalashri_increment/dim_constructors"

In [0]:
constructors_df = (
    spark.read.format("delta")
        .load(constructors_silver_path)
        .filter(F.col("batch_id") == v_batch_id)
)

ref_nationality_region_df = (
    spark.read.format("delta")
        .load(nationality_region_gold_path)
)

display(constructors_df)
display(ref_nationality_region_df)

## Step 2 - Join constructors with nationality_region_df using nationality
Select the following columns

1. constructors.constructor_id
2. constructors.constructor_name
3. constructors.nationality
4. ref_nationality_region.region

In [0]:
dim_constructors_df = (
    constructors_df
        .join(
            ref_nationality_region_df,
            constructors_df.nationality == ref_nationality_region_df.nationality,
            "left"
            )
        .select (
            constructors_df.constructor_id,
            constructors_df.constructor_name,
            constructors_df.nationality,
            ref_nationality_region_df.region.alias("nationality_region")
        )
)

display(dim_constructors_df)

## Step 3 - Write the transformed data to the gold dim_constructors table

In [0]:
write_to_gold(
    input_df=dim_constructors_df,
    target_path=gold_path,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Gold/Kalashri_increment/dim_constructors`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Gold/Kalashri_increment/dim_constructors",
#     True
# )